# Business Data Import, Data Types, and Analytical Structures

This notebook is a step-by-step practice guide for importing enterprise data from **CSV, Excel, a database, and an API**, then validating data types and organizing analytical structures.

**Dataset:** `business_sales_600.csv` / `business_sales_600.xlsx` / `business_sales.db`
**Rows:** 600 transactions
**Business context:** a multi-channel enterprise selling products and services across regions.

## Learning objectives

By the end of the notebook you should be able to:
1. Import tabular data from CSV and Excel.
2. Query transaction data from SQLite with SQL.
3. Retrieve JSON from an HTTP API and normalize it into a DataFrame.
4. Distinguish identifiers, dates, categorical dimensions, measures, flags, and free text.
5. Build clean analytical structures such as a fact table, dimension-like tables, and grouped summaries.

## 0. Setup
Install missing packages in your environment before starting. The core workflow uses pandas, numpy, openpyxl, requests, and sqlite3.

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import json
import requests
from pathlib import Path

DATA_DIR = Path(r'C:\Users\admin\Documents\School\PythonFiles')
CSV_PATH = DATA_DIR / 'business_sales_600.csv'
XLSX_PATH = DATA_DIR / 'business_sales_600.xlsx'
DB_PATH = DATA_DIR / 'business_sales.db'
API_JSON_PATH = DATA_DIR / 'sales_api_sample.json'

print('Files:', CSV_PATH, XLSX_PATH, DB_PATH, API_JSON_PATH)

Files: C:\Users\admin\Documents\School\PythonFiles\business_sales_600.csv C:\Users\admin\Documents\School\PythonFiles\business_sales_600.xlsx C:\Users\admin\Documents\School\PythonFiles\business_sales.db C:\Users\admin\Documents\School\PythonFiles\sales_api_sample.json


## 1. Import CSV data
CSV is a common exchange format. Start with a raw load, inspect the first rows, and confirm the shape.

In [5]:
sales_csv = pd.read_csv(CSV_PATH)
print(sales_csv.shape)
sales_csv.head(10)

(600, 21)


,Order_ID,Order_Date,Customer_ID,Customer_Segment,Region,Product_Category,Product_Name,Sales_Channel,Quantity,Unit_Price,...,Revenue,Cost,Profit,Payment_Method,Customer_Rating,Is_Returned,Sales_Rep,Delivery_Days,Campaign_Source,Notes
0,ORD-100001,2025-11-24,CUST-1027,Corporate,Central,Technology,Laptop Pro 14,Online,12,1266.41,...,14437.02,11667.59,2769.43,Purchase Order,4.6,False,A. Cruz,3,Email,New account
1,ORD-100002,2025-02-17,CUST-1115,Corporate,East,Office Supplies,Ink Cartridge Pack,Online,9,120.13,...,918.96,567.40,351.56,Bank Transfer,5.0,True,G. Ramos,8,Social,NaN
2,ORD-100003,2025-05-23,CUST-1089,Corporate,East,Office Supplies,Ink Cartridge Pack,Online,6,117.60,...,705.59,555.92,149.67,Purchase Order,5.0,False,B. Santos,2,Referral,NaN
3,ORD-100004,2025-02-10,CUST-1059,Small Business,North,Furniture,Storage Shelf,Retail Store,6,270.33,...,1378.69,1120.57,258.12,Credit Card,4.7,False,G. Ramos,2,Social,NaN
4,ORD-100005,2025-08-21,CUST-1019,Corporate,South,Furniture,Standing Desk,Retail Store,6,670.82,...,3823.68,2412.60,1411.09,Bank Transfer,3.8,False,H. Navarro,5,Referral,New account
5,ORD-100006,2025-05-19,CUST-1069,Consumer,South,Office Supplies,Ergonomic Notebook,Corporate Sales,1,17.30,...,17.30,12.21,5.09,Bank Transfer,3.8,False,H. Navarro,6,Referral,Priority client
6,ORD-100007,2025-11-26,CUST-1068,Corporate,Central,Services,Installation Service,Partner,5,246.60,...,1232.99,801.31,431.67,Purchase Order,3.3,False,F. Tan,9,Search,Follow-up required
7,ORD-100008,2025-03-12,CUST-1109,Corporate,North,Services,Consulting Hours,Partner,1,172.52,...,172.52,115.23,57.29,Purchase Order,4.3,False,H. Navarro,3,Trade Show,NaN
8,ORD-100009,2025-05-09,CUST-1076,Small Business,South,Technology,Laptop Pro 14,Partner,11,1287.59,...,12038.94,9251.23,2787.71,Credit Card,4.2,False,E. Garcia,6,Trade Show,NaN
9,ORD-100010,2025-04-02,CUST-1042,Corporate,Central,Technology,Wireless Headset,Online,11,151.75,...,1418.90,838.78,580.12,Mobile Wallet,3.0,False,H. Navarro,3,Email,NaN


In [6]:
# Inspect columns and preliminary pandas dtypes
sales_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          600 non-null    str    
 1   Order_Date        600 non-null    str    
 2   Customer_ID       600 non-null    str    
 3   Customer_Segment  600 non-null    str    
 4   Region            600 non-null    str    
 5   Product_Category  600 non-null    str    
 6   Product_Name      600 non-null    str    
 7   Sales_Channel     600 non-null    str    
 8   Quantity          600 non-null    int64  
 9   Unit_Price        600 non-null    float64
 10  Discount_Rate     600 non-null    float64
 11  Revenue           600 non-null    float64
 12  Cost              600 non-null    float64
 13  Profit            600 non-null    float64
 14  Payment_Method    600 non-null    str    
 15  Customer_Rating   591 non-null    float64
 16  Is_Returned       600 non-null    bool   
 17  Sales_Re

### 1.1 Convert dates explicitly
Date columns are critical for time-series analysis. Do not rely on inference alone when reproducibility matters.

In [7]:
sales_csv['Order_Date'] = pd.to_datetime(sales_csv['Order_Date'], errors='coerce')
print(sales_csv['Order_Date'].dtype)
print(sales_csv['Order_Date'].min(), 'to', sales_csv['Order_Date'].max())

datetime64[us]
2025-01-03 00:00:00 to 2025-12-31 00:00:00


## 2. Import Excel data
Excel is useful when analysts need worksheets, business formatting, or multi-tab workbooks.

In [9]:
sales_xlsx = pd.read_excel(XLSX_PATH, sheet_name='Sales Data')
print(sales_xlsx.shape)
sales_xlsx.head(7)

(600, 21)


,Order_ID,Order_Date,Customer_ID,Customer_Segment,Region,Product_Category,Product_Name,Sales_Channel,Quantity,Unit_Price,...,Revenue,Cost,Profit,Payment_Method,Customer_Rating,Is_Returned,Sales_Rep,Delivery_Days,Campaign_Source,Notes
0,ORD-100001,2025-11-24,CUST-1027,Corporate,Central,Technology,Laptop Pro 14,Online,12,1266.41,...,14437.02,11667.59,2769.43,Purchase Order,4.6,False,A. Cruz,3,Email,New account
1,ORD-100002,2025-02-17,CUST-1115,Corporate,East,Office Supplies,Ink Cartridge Pack,Online,9,120.13,...,918.96,567.40,351.56,Bank Transfer,5.0,True,G. Ramos,8,Social,NaN
2,ORD-100003,2025-05-23,CUST-1089,Corporate,East,Office Supplies,Ink Cartridge Pack,Online,6,117.60,...,705.59,555.92,149.67,Purchase Order,5.0,False,B. Santos,2,Referral,NaN
3,ORD-100004,2025-02-10,CUST-1059,Small Business,North,Furniture,Storage Shelf,Retail Store,6,270.33,...,1378.69,1120.57,258.12,Credit Card,4.7,False,G. Ramos,2,Social,NaN
4,ORD-100005,2025-08-21,CUST-1019,Corporate,South,Furniture,Standing Desk,Retail Store,6,670.82,...,3823.68,2412.60,1411.09,Bank Transfer,3.8,False,H. Navarro,5,Referral,New account
5,ORD-100006,2025-05-19,CUST-1069,Consumer,South,Office Supplies,Ergonomic Notebook,Corporate Sales,1,17.30,...,17.30,12.21,5.09,Bank Transfer,3.8,False,H. Navarro,6,Referral,Priority client
6,ORD-100007,2025-11-26,CUST-1068,Corporate,Central,Services,Installation Service,Partner,5,246.60,...,1232.99,801.31,431.67,Purchase Order,3.3,False,F. Tan,9,Search,Follow-up required


In [10]:
# Load the data dictionary sheet too
data_dictionary = pd.read_excel(XLSX_PATH, sheet_name='Data Dictionary')
data_dictionary.head(10)

,Variable,Type,Role,Description
0,Order_ID,String,Identifier,Unique transaction identifier
1,Order_Date,Date,Time,Transaction date
2,Customer_ID,String,Key,Customer reference key
3,Customer_Segment,Categorical,Dimension,Customer grouping
4,Region,Categorical,Dimension,Sales territory
5,Product_Category,Categorical,Dimension,Product family
6,Product_Name,Categorical,Dimension,Specific product/service
7,Sales_Channel,Categorical,Dimension,Route to market
8,Quantity,Integer,Measure,Units sold
9,Unit_Price,Float,Measure,Price per unit


## 3. Import from a database
Databases are better suited to larger, shared, or governed data. This exercise uses SQLite so the notebook remains self-contained.

In [11]:
conn = sqlite3.connect(DB_PATH)

# List available tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
tables

,name
0,customers
1,products
2,sales_transactions


In [12]:
# Pull only the fields and rows needed for analysis
sales_db = pd.read_sql_query("""SELECT Order_ID, Order_Date, Customer_ID, Region, Product_Category, Revenue, Profit
FROM sales_transactions
WHERE Revenue > 500
ORDER BY Order_Date DESC
LIMIT 20""", conn)
sales_db.head()

,Order_ID,Order_Date,Customer_ID,Region,Product_Category,Revenue,Profit
0,ORD-100559,2025-12-31,CUST-1042,South,Services,11495.78,4164.16
1,ORD-100202,2025-12-30,CUST-1009,South,Services,3364.83,689.04
2,ORD-100261,2025-12-30,CUST-1015,North,Services,2017.89,627.99
3,ORD-100025,2025-12-28,CUST-1010,Central,Services,914.09,312.58
4,ORD-100157,2025-12-27,CUST-1138,South,Furniture,4982.06,1637.04


In [13]:
# Example analytical join across transaction + customer dimensions
joined = pd.read_sql_query("""
SELECT s.Order_ID, s.Order_Date, s.Customer_ID, c.Industry, c.Annual_Spend_Band,
       s.Product_Category, s.Revenue, s.Profit
FROM sales_transactions AS s
LEFT JOIN customers AS c ON s.Customer_ID = c.Customer_ID
LIMIT 25
""", conn)
joined.head()

,Order_ID,Order_Date,Customer_ID,Industry,Annual_Spend_Band,Product_Category,Revenue,Profit
0,ORD-100001,2025-11-24,CUST-1027,Manufacturing,50K-100K,Technology,14437.02,2769.43
1,ORD-100002,2025-02-17,CUST-1115,Financial Services,100K-250K,Office Supplies,918.96,351.56
2,ORD-100003,2025-05-23,CUST-1089,Hospitality,50K-100K,Office Supplies,705.59,149.67
3,ORD-100004,2025-02-10,CUST-1059,Education,50K-100K,Furniture,1378.69,258.12
4,ORD-100005,2025-08-21,CUST-1019,Hospitality,100K-250K,Furniture,3823.68,1411.09


## 4. Import API data
APIs usually return JSON. This section demonstrates the HTTP pattern with a tiny local HTTP server that serves the provided business sample, so the notebook works without an external service.

In [ ]:
# Start a local HTTP server in a background thread.
# The API reads sales_api_sample.json from the current directory.
import threading
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

server = ThreadingHTTPServer(('127.0.0.1', 8000), QuietHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print('Local API running at http://127.0.0.1:8000/sales_api_sample.json')

In [ ]:
response = requests.get('http://127.0.0.1:8000/sales_api_sample.json', timeout=10)
response.raise_for_status()
payload = response.json()
print(payload['status'], payload['count'])
api_sales = pd.json_normalize(payload['data'])
api_sales.head()

In [ ]:
api_sales['Order_Date'] = pd.to_datetime(api_sales['Order_Date'], errors='coerce')
api_sales['Revenue'] = pd.to_numeric(api_sales['Revenue'], errors='coerce')
api_sales.dtypes

### 4.1 Generic external API template
Replace the URL and authentication method with the API documentation for your source.

In [15]:
API_URL = 'https://example.com/api/v1/sales'
HEADERS = {'Authorization': 'Bearer YOUR_TOKEN'}

# Example pattern only - do not run until the endpoint and token are configured.
# response = requests.get(API_URL, headers=HEADERS, params={'limit': 100}, timeout=30)
# response.raise_for_status()
# api_df = pd.json_normalize(response.json()['data'])

## 5. Data types and variable roles
A variable's **storage type** (string, integer, float, datetime, boolean) is not the same as its **analytical role**. For example, `Customer_ID` is stored as text but acts as a key, while `Revenue` is numeric and acts as a measure.

In [16]:
type_map = pd.DataFrame({
    'Variable': [
        'Order_ID','Order_Date','Customer_ID','Customer_Segment','Region',
        'Product_Category','Product_Name','Sales_Channel','Quantity','Unit_Price',
        'Discount_Rate','Revenue','Cost','Profit','Payment_Method',
        'Customer_Rating','Is_Returned','Sales_Rep','Delivery_Days','Campaign_Source','Notes'
    ],
    'Storage_Type': [
        'string','datetime','string','category','category','category','category','category',
        'integer','float','float','float','float','float','category','float','boolean',
        'category','integer','category','string'
    ],
    'Analytical_Role': [
        'Identifier','Time','Key','Dimension','Dimension','Dimension','Dimension','Dimension',
        'Measure','Measure','Measure','Measure','Measure','Measure','Dimension','Measure',
        'Flag','Dimension','Measure','Dimension','Text'
    ]
})
type_map

,Variable,Storage_Type,Analytical_Role
0,Order_ID,string,Identifier
1,Order_Date,datetime,Time
2,Customer_ID,string,Key
3,Customer_Segment,category,Dimension
4,Region,category,Dimension
5,Product_Category,category,Dimension
6,Product_Name,category,Dimension
7,Sales_Channel,category,Dimension
8,Quantity,integer,Measure
9,Unit_Price,float,Measure


## 6. Apply analytical dtypes intentionally
Categorical columns can use pandas `category` dtype. This can reduce memory and make group-by logic clearer. Use nullable numeric types when missing values are expected.

In [17]:
sales = sales_csv.copy()

cat_cols = [
    'Customer_Segment','Region','Product_Category','Product_Name',
    'Sales_Channel','Payment_Method','Sales_Rep','Campaign_Source'
]
for col in cat_cols:
    sales[col] = sales[col].astype('category')

sales['Quantity'] = pd.to_numeric(sales['Quantity'], errors='coerce').astype('Int64')
sales['Delivery_Days'] = pd.to_numeric(sales['Delivery_Days'], errors='coerce').astype('Int64')
for col in ['Unit_Price','Discount_Rate','Revenue','Cost','Profit','Customer_Rating']:
    sales[col] = pd.to_numeric(sales[col], errors='coerce')
sales['Is_Returned'] = sales['Is_Returned'].astype('boolean')
sales['Order_Date'] = pd.to_datetime(sales['Order_Date'], errors='coerce')

sales.dtypes

Order_ID                       str
Order_Date          datetime64[us]
Customer_ID                    str
Customer_Segment          category
Region                    category
Product_Category          category
Product_Name              category
Sales_Channel             category
Quantity                     Int64
Unit_Price                 float64
Discount_Rate              float64
Revenue                    float64
Cost                       float64
Profit                     float64
Payment_Method            category
Customer_Rating            float64
Is_Returned                boolean
Sales_Rep                 category
Delivery_Days                Int64
Campaign_Source           category
Notes                          str
dtype: object

## 7. Validate data quality before analysis
Check row counts, missing values, duplicates, valid ranges, and key uniqueness.

In [18]:
print('Rows:', len(sales))
print('Duplicate Order_IDs:', sales['Order_ID'].duplicated().sum())
print('Missing values by column:')
print(sales.isna().sum().sort_values(ascending=False).head(10))

Rows: 600
Duplicate Order_IDs: 0
Missing values by column:
Notes               354
Customer_Rating       9
Campaign_Source       7
Order_Date            0
Order_ID              0
Region                0
Customer_Segment      0
Customer_ID           0
Product_Category      0
Unit_Price            0
dtype: int64


In [19]:
print('Negative revenue rows:', (sales['Revenue'] < 0).sum())
print('Rating outside 1-5:', ((sales['Customer_Rating'] < 1) | (sales['Customer_Rating'] > 5)).sum())
print('Discount outside 0-1:', ((sales['Discount_Rate'] < 0) | (sales['Discount_Rate'] > 1)).sum())

Negative revenue rows: 0
Rating outside 1-5: 0
Discount outside 0-1: 0


## 8. Analytical structures
Use structures that match the business question.

- **Fact table:** transaction-level rows with keys and numeric measures.
- **Dimension-like tables:** customer, product, region, channel, date attributes.
- **Grouped summary:** aggregated KPIs by one or more dimensions.
- **Wide analytical table:** convenient for modeling or visualization when each row is an analytical entity.
- **Long/tidy table:** one observation per row-variable combination, useful for many visualization workflows.

In [20]:
# Example fact-table view
fact_sales = sales[['Order_ID','Order_Date','Customer_ID','Product_Category','Product_Name',
                   'Sales_Channel','Region','Quantity','Revenue','Cost','Profit','Is_Returned']].copy()
fact_sales.head()

,Order_ID,Order_Date,Customer_ID,Product_Category,Product_Name,Sales_Channel,Region,Quantity,Revenue,Cost,Profit,Is_Returned
0,ORD-100001,2025-11-24,CUST-1027,Technology,Laptop Pro 14,Online,Central,12,14437.02,11667.59,2769.43,False
1,ORD-100002,2025-02-17,CUST-1115,Office Supplies,Ink Cartridge Pack,Online,East,9,918.96,567.40,351.56,True
2,ORD-100003,2025-05-23,CUST-1089,Office Supplies,Ink Cartridge Pack,Online,East,6,705.59,555.92,149.67,False
3,ORD-100004,2025-02-10,CUST-1059,Furniture,Storage Shelf,Retail Store,North,6,1378.69,1120.57,258.12,False
4,ORD-100005,2025-08-21,CUST-1019,Furniture,Standing Desk,Retail Store,South,6,3823.68,2412.60,1411.09,False


In [21]:
# Dimension-like customer table
dim_customer = pd.read_sql_query(
    'SELECT * FROM customers',
    conn
).drop_duplicates('Customer_ID')

# Dimension-like product table
dim_product = pd.read_sql_query(
    'SELECT * FROM products',
    conn
).drop_duplicates('Product_Name')

dim_customer.head(), dim_product.head()

(  Customer_ID  Customer_Name            Industry Annual_Spend_Band
 0   CUST-1001  Customer 1001           Education         100K-250K
 1   CUST-1002  Customer 1002  Financial Services             250K+
 2   CUST-1003  Customer 1003       Manufacturing              <50K
 3   CUST-1004  Customer 1004           Education         100K-250K
 4   CUST-1005  Customer 1005              Retail         100K-250K,
            Product_Name Product_Category  List_Price
 0         Laptop Pro 14       Technology        1299
 1       Business Tablet       Technology         699
 2      Wireless Headset       Technology         149
 3            4K Monitor       Technology         459
 4  Cloud Security Suite       Technology         899)

In [22]:
# Build a date dimension
date_dim = pd.DataFrame({'Order_Date': pd.date_range(sales['Order_Date'].min(), sales['Order_Date'].max(), freq='D')})
date_dim['Year'] = date_dim['Order_Date'].dt.year
date_dim['Quarter'] = date_dim['Order_Date'].dt.quarter
date_dim['Month'] = date_dim['Order_Date'].dt.month
date_dim['Month_Name'] = date_dim['Order_Date'].dt.month_name()
date_dim['Weekday'] = date_dim['Order_Date'].dt.day_name()
date_dim.head()

,Order_Date,Year,Quarter,Month,Month_Name,Weekday
0,2025-01-03,2025,1,1,January,Friday
1,2025-01-04,2025,1,1,January,Saturday
2,2025-01-05,2025,1,1,January,Sunday
3,2025-01-06,2025,1,1,January,Monday
4,2025-01-07,2025,1,1,January,Tuesday


## 9. Create business summaries
Aggregation turns transaction records into decision-ready metrics.

In [23]:
regional_summary = (
    sales.groupby('Region', observed=True)
         .agg(Orders=('Order_ID','nunique'),
              Revenue=('Revenue','sum'),
              Profit=('Profit','sum'),
              Avg_Rating=('Customer_Rating','mean'))
         .sort_values('Revenue', ascending=False)
)
regional_summary

,Orders,Revenue,Profit,Avg_Rating
Region,,,,
North,120,337597.56,103109.67,4.064167
Central,111,334133.01,104191.70,4.133945
East,132,333022.34,100771.46,4.210078
West,119,329021.34,95252.58,4.113559
South,118,324314.76,100142.11,4.032174


In [ ]:
category_channel = (
    sales.pivot_table(index='Product_Category', columns='Sales_Channel',
                      values='Revenue', aggfunc='sum', observed=True)
         .round(2)
)
category_channel

## 10. Export a cleaned analytical table
Once types and quality checks are complete, save a cleaned version for downstream work.

In [ ]:
clean_path = DATA_DIR / 'business_sales_clean.csv'
sales.to_csv(clean_path, index=False)
print('Saved:', clean_path)

## Practice exercises

1. Import the CSV and identify all numeric columns.
2. Find the top 10 customers by profit.
3. Compare revenue and return rate by sales channel.
4. Add `Year` and `Month` from `Order_Date` and summarize monthly revenue.
5. Join the transaction fact table to the customer table and compare revenue by industry.
6. Modify the API pattern to retrieve paginated data from a real business API available to you.

## Key takeaways

Reliable analytics starts with reliable data import. Always inspect the source, assign explicit types, preserve identifiers as keys, separate dimensions from measures, validate quality, and then aggregate into structures that match the business question.